## Purpose 

This script creates a model for predicting water temperature in Shadow Mountain Reservoir using a random test set of two years of data. Directions for creating the virtual environment are includeed in the README.md file.

### Import Modules

In [1]:
#high level modules
import os
import imp
import pandas as pd

# ml/ai modules
import tensorflow as tf

/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_79377/2087554530.py:3: DeprecationWarning: the imp module is deprecated in favour of importlib and slated for removal in Python 3.12; see the module's documentation for alternative uses
  import imp


### Custom Modules

In [2]:
# import custom modules
this_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/baseline_model/"
imp.load_source("settings",os.path.join(this_dir,"settings.py"))
from settings import settings
imp.load_source("architecture", os.path.join(this_dir, "architecture.py"))
from architecture import build_model, compile_model
imp.load_source("universals", os.path.join(this_dir, "universal_functions.py"))
from universals import save_to_pickle, twotemp_labels_features

# point to data directory
data_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/baseline_model/data/"

### Import train/val sets

Import and format training and validation arrays for use in model training

In [3]:
all_files = pd.Series(os.listdir(data_dir))
test = all_files[all_files.str.contains('test')]
val = all_files[all_files.str.contains('validation')]
train = all_files[all_files.str.contains('training')]

# these files end up in no particular order, so we need to sort them
validation_files = val.sort_values()
training_files = train.sort_values()

# function to load files
def load_data(file):
    return pd.read_csv(os.path.join(data_dir, file), sep=',')

val1 = load_data(validation_files.values[0])
train1 = load_data(training_files.values[0])

val2 = load_data(validation_files.values[1])
train2 = load_data(training_files.values[1])

val3 = load_data(validation_files.values[2])
train3 = load_data(training_files.values[2])

val4 = load_data(validation_files.values[3])
train4 = load_data(training_files.values[3])


In [4]:
val1.columns

Index(['date', 'mean_1m_temp_degC', 'mean_0_5m_temp_degC',
       'mean_1m_temp_degC_m1', 'mean_0_5m_temp_degC_m1',
       'mean_1m_temp_degC_m2', 'mean_0_5m_temp_degC_m2',
       'mean_1m_temp_degC_m3', 'mean_0_5m_temp_degC_m3', 'mean_air_temp',
       'max_air_temp', 'min_air_temp', 'mean_wind', 'max_wind', 'min_wind',
       'mean_rel_hum', 'max_rel_hum', 'total_solar_radiation',
       'mean_air_temp_m1', 'max_air_temp_m1', 'min_air_temp_m1',
       'mean_wind_m1', 'max_wind_m1', 'min_wind_m1', 'mean_rel_hum_m1',
       'max_rel_hum_m1', 'total_solar_radiation_m1', 'mean_air_temp_m2',
       'max_air_temp_m2', 'min_air_temp_m2', 'mean_wind_m2', 'max_wind_m2',
       'min_wind_m2', 'mean_rel_hum_m2', 'max_rel_hum_m2',
       'total_solar_radiation_m2', 'mean_air_temp_m3', 'max_air_temp_m3',
       'min_air_temp_m3', 'mean_wind_m3', 'max_wind_m3', 'min_wind_m3',
       'mean_rel_hum_m3', 'max_rel_hum_m3', 'total_solar_radiation_m3',
       'mean_air_temp_m4', 'max_air_temp_m4', 'min_

Using the function twotemp_labels_features, we can create ML-ready features and labels for the training and validation sets.

In [5]:
features1, labels_1, val_features1, val_labels_1 = twotemp_labels_features(train1, val1)
features2, labels_2, val_features2, val_labels_2 = twotemp_labels_features(train2, val2)
features3, labels_3, val_features3, val_labels_3 = twotemp_labels_features(train3, val3)
features4, labels_4, val_features4, val_labels_4 = twotemp_labels_features(train4, val4)

### Compile and train models

Look at available settings for the model.

In [6]:
settings.keys()

dict_keys(['overfit', 'simple', 'three_ten', 'three_ten_larger_eta', 'five_ten', 'five_ten_larger_eta'])

make sure the setting is working correctly

In [7]:
model_setting = "three_ten"
print(settings[model_setting]["random_seed"])

57


In [8]:


tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(settings[model_setting]["random_seed"])

# define the early stopping callback
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
  monitor="val_loss", 
  patience=settings[model_setting]["patience"], 
  restore_best_weights=True, 
  mode="auto"
)

## TS cross 1
model_1 = build_model(
  features1, 
  labels_1, 
  settings[model_setting])

model_1 = compile_model(
  model_1, 
  settings[model_setting])

# train the model via model.fit
history_1 = model_1.fit(
  features1, 
  labels_1, 
  epochs=settings[model_setting]["max_epochs"],
  batch_size=settings[model_setting]["batch_size"],
  shuffle=True,
  validation_data=[val_features1, val_labels_1],
  callbacks=[early_stopping_callback],
  verbose=1,
)

## TS cross 2
model_2 = build_model(
  features2,
  labels_2, 
  settings[model_setting])
model_2 = compile_model(model_2, settings[model_setting])

# train the model via model.fit
history_2 = model_2.fit(
  features2,
  labels_2,
  epochs=settings[model_setting]["max_epochs"],
  batch_size=settings[model_setting]["batch_size"],
  shuffle=True,
  validation_data=[val_features2, val_labels_2],
  callbacks=[early_stopping_callback],
  verbose=1,
)

## TS cross 3

model_3 = build_model(
  features3,
  labels_3,
  settings[model_setting])
model_3 = compile_model(model_3, settings[model_setting])

# train the model via model.fit
history_3 = model_3.fit(
  features3,
  labels_3,
  epochs=settings[model_setting]["max_epochs"],
  batch_size=settings[model_setting]["batch_size"],
  shuffle=True,
  validation_data=[val_features3, val_labels_3],
  callbacks=[early_stopping_callback],
  verbose=1,
)

## TS cross 4

model_4 = build_model(
  features4,
  labels_4,
  settings[model_setting])
model_4 = compile_model(model_4, settings[model_setting])

# train the model via model.fit
history_4 = model_4.fit(
  features4,
  labels_4,
  epochs=settings[model_setting]["max_epochs"],
  batch_size=settings[model_setting]["batch_size"],
  shuffle=True,
  validation_data=[val_features4, val_labels_4],
  callbacks=[early_stopping_callback],
  verbose=1,
)


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 76)]              0         
                                                                 
 dropout (Dropout)           (None, 76)                0         
                                                                 
 dense (Dense)               (None, 10)                770       
                                                                 
 dense_1 (Dense)             (None, 10)                110       
                                                                 
 dense_2 (Dense)             (None, 10)                110       
                                                                 
 dense_3 (Dense)             (None, 2)                 22        
                                                                 
Total params: 1,012
Trainable params: 1,012
Non-trainable par

2025-05-07 12:22:55.781663: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


10/10 [==============================] - 0s 1ms/step - loss: 1.0078 - val_loss: 0.7217
Epoch 3/1000
10/10 [==============================] - 0s 4ms/step - loss: 0.9815 - val_loss: 0.6903
Epoch 4/1000
10/10 [==============================] - 0s 2ms/step - loss: 0.9183 - val_loss: 0.6261
Epoch 5/1000
10/10 [==============================] - 0s 1ms/step - loss: 0.8038 - val_loss: 0.5249
Epoch 6/1000
10/10 [==============================] - 0s 1ms/step - loss: 0.6430 - val_loss: 0.4006
Epoch 7/1000
10/10 [==============================] - 0s 1ms/step - loss: 0.4686 - val_loss: 0.2807
Epoch 8/1000
10/10 [==============================] - 0s 1ms/step - loss: 0.3275 - val_loss: 0.1993
Epoch 9/1000
10/10 [==============================] - 0s 1ms/step - loss: 0.2430 - val_loss: 0.1696
Epoch 10/1000
10/10 [==============================] - 0s 1ms/step - loss: 0.2005 - val_loss: 0.1510
Epoch 11/1000
10/10 [==============================] - 0s 1ms/step - loss: 0.1751 - val_loss: 0.1359
Epoch 12/10

And save the models and training history - dump directory may need to be adjusted.

In [9]:
dump_dir = os.path.join("/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/baseline_model/dump/", model_setting)
!mkdir -p $dump_dir

# save models to pickle
models = [model_1, model_2, model_3, model_4]

for model, i in zip(models, range(1,5)):
    save_to_pickle(model, f"{dump_dir}/model_{i}.pkl")

# save history to pickles
histories = [history_1, history_2, history_3, history_4]

for history, i in zip(histories, range(1,5)):
    save_to_pickle(history, f"{dump_dir}/history_{i}.pkl")


INFO:tensorflow:Assets written to: ram://bf2702ac-7425-4b46-938d-ade23bf94bad/assets


2025-05-07 12:23:53.460857: W tensorflow/python/util/util.cc:368] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.


INFO:tensorflow:Assets written to: ram://0b2c3a5a-89bb-4da3-9987-d5ec7824f3ad/assets
INFO:tensorflow:Assets written to: ram://02d1b9ce-a754-48d5-bf4c-47912daaa692/assets
INFO:tensorflow:Assets written to: ram://b18a82c1-39b4-4087-991b-c1c9b9ff8633/assets
INFO:tensorflow:Assets written to: ram://0d26ea26-6362-4a45-bcea-5041d5fadb1c/assets
INFO:tensorflow:Assets written to: ram://aed08959-e138-46b5-9988-af1f10ad35ab/assets
INFO:tensorflow:Assets written to: ram://1d9ec6d2-f664-4ff6-a11f-ac92e7d7e176/assets
INFO:tensorflow:Assets written to: ram://86a4c484-a740-4c86-a753-1e56c462a0a6/assets
